# 🤖 Day 24 — Transformer from Scratch, Seq2Seq & Mini LM Capstone
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Full Transformer Architecture from Scratch | ~12 sec |
| 2 | 10:30–11:00 AM | Positional Encoding & Feed-Forward Network | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Seq2Seq Encoder-Decoder + Fine-Tuning | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Mini Transformer Capstone: Sequence Classification | ~18 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_wine, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

# ── Shared utility functions used across all sessions ──────────
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def layer_norm(x, gamma=None, beta=None, eps=1e-6):
    mu    = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1,  keepdims=True)
    x_hat = (x - mu) / (sigma + eps)
    if gamma is not None:
        x_hat = gamma * x_hat
    if beta is not None:
        x_hat = x_hat + beta
    return x_hat

def gelu(x):
    return x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715*x**3))) / 2

print('Utility functions: softmax, layer_norm, gelu — all defined.')

---
## Session 1 — Full Transformer Architecture from Scratch
**Time:** 9:00–10:30 AM | **Run time: ~12 sec**

### Architecture: Attention Is All You Need
```
Input → Embed + PE → [MHA → Add&Norm → FFN → Add&Norm] × N → Output
```

In [ ]:
# 1.1  Scaled Dot-Product Attention
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Attention(Q,K,V) = softmax(QKᵀ / √d_k) · V
    mask: additive mask (-inf blocks positions)
    """
    d_k     = Q.shape[-1]
    scores  = Q @ K.T / np.sqrt(d_k)        # (seq, seq)
    if mask is not None:
        scores = scores + mask               # -inf → 0 after softmax
    weights = softmax(scores, axis=-1)       # attention weights
    output  = weights @ V
    return output, weights

# 1.2  Multi-Head Attention (MHA)
class MultiHeadAttention:
    """
    MHA: split d_model into h heads, run attention in parallel,
    concatenate and project through W_o.
    """
    def __init__(self, d_model, n_heads, seed=42):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        rng_mha = np.random.default_rng(seed)
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        # Projection matrices (one per head, stacked for efficiency)
        self.Wq = rng_mha.normal(0, 0.05, (d_model, d_model))
        self.Wk = rng_mha.normal(0, 0.05, (d_model, d_model))
        self.Wv = rng_mha.normal(0, 0.05, (d_model, d_model))
        self.Wo = rng_mha.normal(0, 0.05, (d_model, d_model))

    def split_heads(self, X):
        """Split last dim into (n_heads, d_k)."""
        seq = X.shape[0]
        return X.reshape(seq, self.n_heads, self.d_k)  # (seq, h, d_k)

    def forward(self, Q_in, K_in, V_in, mask=None):
        Q = Q_in @ self.Wq   # (seq_q, d_model)
        K = K_in @ self.Wk   # (seq_k, d_model)
        V = V_in @ self.Wv   # (seq_k, d_model)

        head_outputs, all_weights = [], []
        for h in range(self.n_heads):
            sl  = slice(h*self.d_k, (h+1)*self.d_k)
            out_h, w_h = scaled_dot_product_attention(
                Q[:, sl], K[:, sl], V[:, sl], mask=mask
            )
            head_outputs.append(out_h)
            all_weights.append(w_h)

        concat = np.hstack(head_outputs)   # (seq, d_model)
        output = concat @ self.Wo          # final projection
        return output, np.array(all_weights)

# Demo
rng = np.random.default_rng(42)
SEQ_LEN, D_MODEL, N_HEADS = 8, 32, 4
X_demo = rng.normal(0, 1, (SEQ_LEN, D_MODEL))
mha    = MultiHeadAttention(d_model=D_MODEL, n_heads=N_HEADS)
out_mha, weights_mha = mha.forward(X_demo, X_demo, X_demo)
print(f'Multi-Head Attention:')
print(f'  Input  : {X_demo.shape}  (seq={SEQ_LEN}, d={D_MODEL})')
print(f'  Output : {out_mha.shape}')
print(f'  Weights: {weights_mha.shape}  ({N_HEADS} heads × {SEQ_LEN} × {SEQ_LEN})')
print(f'  Row sums (must be 1): {weights_mha[0].sum(axis=-1).round(4)}')

In [ ]:
# 1.3  Feed-Forward Network (FFN)
class FeedForward:
    """
    FFN(x) = GELU(x W_1 + b_1) W_2 + b_2
    d_ff = 4 × d_model (expand then contract)
    """
    def __init__(self, d_model, d_ff=None, seed=42):
        rng_ff  = np.random.default_rng(seed)
        d_ff    = d_ff or d_model * 4
        self.W1 = rng_ff.normal(0, 0.02, (d_model, d_ff))
        self.b1 = np.zeros(d_ff)
        self.W2 = rng_ff.normal(0, 0.02, (d_ff, d_model))
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        h = gelu(x @ self.W1 + self.b1)   # expand
        return h @ self.W2 + self.b2       # contract

# Demo
ffn_layer = FeedForward(d_model=D_MODEL, d_ff=D_MODEL*4)
ffn_out   = ffn_layer.forward(out_mha)
print(f'\nFeed-Forward Network:')
print(f'  Input  : {out_mha.shape}')
print(f'  Hidden : {D_MODEL*4}  (4× expansion)')
print(f'  Output : {ffn_out.shape}  (back to d_model)')
print(f'  GELU is smooth: f(0)={gelu(0):.4f}  f(1)={gelu(1.0):.4f}  f(-1)={gelu(-1.0):.4f}')

In [ ]:
# 1.4  Full Transformer Encoder Block
class TransformerEncoderBlock:
    """
    Pre-LN Transformer Encoder Block:
    x = x + MHA(LayerNorm(x))
    x = x + FFN(LayerNorm(x))
    """
    def __init__(self, d_model, n_heads, d_ff=None, seed=42):
        self.mha = MultiHeadAttention(d_model, n_heads, seed=seed)
        self.ffn = FeedForward(d_model, d_ff, seed=seed+1)
        self.gamma1 = np.ones(d_model)
        self.beta1  = np.zeros(d_model)
        self.gamma2 = np.ones(d_model)
        self.beta2  = np.zeros(d_model)

    def forward(self, x, mask=None):
        # Sub-layer 1: MHA with Pre-LN
        x_norm1    = layer_norm(x, self.gamma1, self.beta1)
        mha_out, w = self.mha.forward(x_norm1, x_norm1, x_norm1, mask=mask)
        x          = x + mha_out             # residual connection
        # Sub-layer 2: FFN with Pre-LN
        x_norm2    = layer_norm(x, self.gamma2, self.beta2)
        ffn_out    = self.ffn.forward(x_norm2)
        x          = x + ffn_out             # residual connection
        return x, w

# Stack 2 encoder blocks
block1 = TransformerEncoderBlock(D_MODEL, N_HEADS, seed=42)
block2 = TransformerEncoderBlock(D_MODEL, N_HEADS, seed=99)

h1, w1 = block1.forward(X_demo)
h2, w2 = block2.forward(h1)

print(f'\nStacked Transformer Encoder (2 blocks):')
print(f'  After block 1: {h1.shape}  mean={h1.mean():.4f}  std={h1.std():.4f}')
print(f'  After block 2: {h2.shape}  mean={h2.mean():.4f}  std={h2.std():.4f}')
print(f'  Residual kept scale reasonable: {np.allclose(h2.std(), h1.std(), rtol=2.0)}')

---
## Session 2 — Positional Encoding & Feed-Forward Network
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Sinusoidal Positional Encoding
def sinusoidal_pe(seq_len, d_model):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    Properties: unique per position, consistent relative distances
    """
    PE  = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]
    div = np.power(10000.0, np.arange(0, d_model, 2) / d_model)
    PE[:, 0::2] = np.sin(pos / div)
    PE[:, 1::2] = np.cos(pos / div[:d_model//2])
    return PE

# Visualise PE
SEQ_VIS, D_VIS = 20, 64
pe_matrix = sinusoidal_pe(SEQ_VIS, D_VIS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im = axes[0].imshow(pe_matrix, aspect='auto', cmap='RdBu')
axes[0].set_title('Sinusoidal Positional Encoding')
axes[0].set_xlabel('Embedding dimension'); axes[0].set_ylabel('Token position')
plt.colorbar(im, ax=axes[0])

# Show dot-product similarity between positions (should be similar for close positions)
sim = pe_matrix @ pe_matrix.T
im2 = axes[1].imshow(sim, cmap='Blues')
axes[1].set_title('Position Similarity Matrix (PE dot products)')
axes[1].set_xlabel('Position j'); axes[1].set_ylabel('Position i')
plt.colorbar(im2, ax=axes[1])
plt.suptitle('Sinusoidal PE: encoding and position similarity', fontsize=11)
plt.tight_layout(); plt.show()

print(f'Sinusoidal PE shape: {pe_matrix.shape}')
print(f'Adjacent similarity (pos 0 vs 1): {pe_matrix[0] @ pe_matrix[1] / (np.linalg.norm(pe_matrix[0])*np.linalg.norm(pe_matrix[1])):.4f}')
print(f'Distant similarity (pos 0 vs 19): {pe_matrix[0] @ pe_matrix[19]/(np.linalg.norm(pe_matrix[0])*np.linalg.norm(pe_matrix[19])):.4f}')

In [ ]:
# 2.2  Learned Positional Embedding + GELU Comparison
rng_pe = np.random.default_rng(42)

# Learned PE: just a trainable matrix
learned_pe = rng_pe.normal(0, 0.02, (SEQ_VIS, D_VIS))

print('Positional Encoding Comparison:')
print(f'  Sinusoidal PE: fixed, no parameters, generalises to longer sequences')
print(f'  Learned PE   : {learned_pe.size} trainable parameters, better for fixed-length tasks')

# GELU vs ReLU comparison
x_range = np.linspace(-3, 3, 100)
gelu_vals = gelu(x_range)
relu_vals = np.maximum(0, x_range)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_range, relu_vals, color='#D85A30', linewidth=2, label='ReLU')
axes[0].plot(x_range, gelu_vals, color='#534AB7', linewidth=2, label='GELU', linestyle='--')
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].set_title('ReLU vs GELU Activation')
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x_range, gelu_vals - relu_vals, color='#1D9E75', linewidth=2)
axes[1].set_title('GELU - ReLU difference')
axes[1].set_xlabel('x'); axes[1].set_ylabel('Difference')
axes[1].axhline(0, color='gray', linewidth=0.5); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\nGELU properties:')
print(f'  GELU(0)  = {gelu(0):.4f}  (slight negative region unlike ReLU)')
print(f'  GELU(1)  = {gelu(1.0):.4f}  vs ReLU(1)={max(0,1.0)}')
print(f'  GELU(-1) = {gelu(-1.0):.4f}  vs ReLU(-1)={max(0,-1.0)}')

---
## Session 3 — Seq2Seq Encoder-Decoder + Fine-Tuning Patterns
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Cross-Attention: Q from Decoder, K/V from Encoder
class CrossAttention:
    """
    Decoder cross-attends to encoder output:
    - Q comes from decoder hidden state
    - K, V come from encoder output (memory)
    Each decoder position can look at ALL encoder positions.
    """
    def __init__(self, d_model, n_heads, seed=42):
        rng_ca  = np.random.default_rng(seed)
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        self.Wq = rng_ca.normal(0, 0.05, (d_model, d_model))
        self.Wk = rng_ca.normal(0, 0.05, (d_model, d_model))
        self.Wv = rng_ca.normal(0, 0.05, (d_model, d_model))
        self.Wo = rng_ca.normal(0, 0.05, (d_model, d_model))

    def forward(self, decoder_query, encoder_memory):
        Q = decoder_query  @ self.Wq   # queries from decoder
        K = encoder_memory @ self.Wk   # keys from encoder
        V = encoder_memory @ self.Wv   # values from encoder

        head_outputs = []
        for h in range(self.n_heads):
            sl = slice(h*self.d_k, (h+1)*self.d_k)
            out_h, _ = scaled_dot_product_attention(Q[:,sl], K[:,sl], V[:,sl])
            head_outputs.append(out_h)

        concat = np.hstack(head_outputs)
        return concat @ self.Wo

# Demo: 4-token decoder attending over 6-token encoder
rng = np.random.default_rng(42)
D, SRC_LEN, TGT_LEN = 32, 6, 4
encoder_memory  = rng.normal(0, 1, (SRC_LEN, D))
decoder_queries = rng.normal(0, 1, (TGT_LEN, D))

cross_attn  = CrossAttention(d_model=D, n_heads=4)
cross_out   = cross_attn.forward(decoder_queries, encoder_memory)

print('Cross-Attention (Decoder → Encoder):')
print(f'  Encoder output (memory): {encoder_memory.shape}  (src_len={SRC_LEN}, d={D})')
print(f'  Decoder queries        : {decoder_queries.shape}  (tgt_len={TGT_LEN}, d={D})')
print(f'  Cross-attention output : {cross_out.shape}  (tgt_len={TGT_LEN}, d={D})')

In [ ]:
# 3.2  Full Decoder Block: Masked Self-Attn + Cross-Attn + FFN
class TransformerDecoderBlock:
    """
    Decoder block:
    1. Masked self-attention (causal: can't peek at future tokens)
    2. Cross-attention (attend to encoder memory)
    3. Feed-forward
    All with Pre-LN and residual connections.
    """
    def __init__(self, d_model, n_heads, seed=42):
        self.self_attn  = MultiHeadAttention(d_model, n_heads, seed=seed)
        self.cross_attn = CrossAttention(d_model, n_heads, seed=seed+10)
        self.ffn        = FeedForward(d_model, seed=seed+20)
        self.gammas     = [np.ones(d_model)  for _ in range(3)]
        self.betas      = [np.zeros(d_model) for _ in range(3)]

    def causal_mask(self, n):
        return np.triu(np.ones((n,n))*-1e9, k=1)

    def forward(self, x_tgt, enc_memory):
        # 1. Masked self-attention
        mask     = self.causal_mask(len(x_tgt))
        x_norm1  = layer_norm(x_tgt, self.gammas[0], self.betas[0])
        sa_out,_ = self.self_attn.forward(x_norm1, x_norm1, x_norm1, mask=mask)
        x_tgt    = x_tgt + sa_out
        # 2. Cross-attention
        x_norm2  = layer_norm(x_tgt, self.gammas[1], self.betas[1])
        ca_out   = self.cross_attn.forward(x_norm2, enc_memory)
        x_tgt    = x_tgt + ca_out
        # 3. FFN
        x_norm3  = layer_norm(x_tgt, self.gammas[2], self.betas[2])
        x_tgt    = x_tgt + self.ffn.forward(x_norm3)
        return x_tgt

# Full Encoder-Decoder forward pass
encoder_blk = TransformerEncoderBlock(D, n_heads=4, seed=42)
decoder_blk = TransformerDecoderBlock(D, n_heads=4, seed=42)

X_src = rng.normal(0, 1, (SRC_LEN, D)) + sinusoidal_pe(SRC_LEN, D)
X_tgt = rng.normal(0, 1, (TGT_LEN, D)) + sinusoidal_pe(TGT_LEN, D)

enc_out, enc_weights  = encoder_blk.forward(X_src)
dec_out               = decoder_blk.forward(X_tgt, enc_out)

print(f'\nFull Encoder-Decoder (Seq2Seq):')
print(f'  Encoder input : {X_src.shape}  → output: {enc_out.shape}')
print(f'  Decoder input : {X_tgt.shape}  → output: {dec_out.shape}')
print(f'  Enc→Dec info flow: cross-attention over all {SRC_LEN} encoder positions')

In [ ]:
# 3.3  LoRA Fine-Tuning Pattern
class LoRALayer:
    """
    LoRA: W_eff = W_frozen + (α/r) * A @ B
    Only A and B are trained; W_frozen is kept fixed.
    Parameter efficiency: (d_in + d_out)*r vs d_in*d_out
    """
    def __init__(self, d_in, d_out, rank=4, alpha=16, seed=42):
        rng_l       = np.random.default_rng(seed)
        self.W      = rng_l.normal(0, 0.02, (d_in, d_out))   # frozen
        self.A      = rng_l.normal(0, 0.01, (d_in, rank))    # trainable
        self.B      = np.zeros((rank, d_out))                  # init to 0
        self.scale  = alpha / rank
        self.d_in   = d_in
        self.d_out  = d_out
        self.rank   = rank

    def forward(self, x):
        return x @ self.W + (x @ self.A @ self.B) * self.scale

    def n_trainable(self):
        return self.A.size + self.B.size

    def n_total(self):
        return self.W.size + self.A.size + self.B.size

# Simulate LoRA fine-tuning on a simple regression task
rng_lora = np.random.default_rng(42)
D_LR = 32; RANK = 4
lora = LoRALayer(d_in=D_LR, d_out=D_LR, rank=RANK, alpha=16)

# Target: make the layer compute identity (x → x)
X_lr     = rng_lora.normal(0, 1, (100, D_LR))
y_target = X_lr.copy()  # identity task

loss_history = []
for step in range(200):
    y_pred = lora.forward(X_lr)
    err    = y_pred - y_target
    loss   = np.mean(err**2)
    # Gradients only through A and B
    h_a    = X_lr @ lora.A                        # (n, rank)
    dB     = h_a.T @ err * lora.scale / len(X_lr)
    dA     = X_lr.T @ (err @ lora.B.T * lora.scale) / len(X_lr)
    lora.B -= 0.005 * dB
    lora.A -= 0.005 * dA
    loss_history.append(loss)

print(f'LoRA Fine-Tuning (identity task, rank={RANK}):')
print(f'  Total params    : {lora.n_total():,}')
print(f'  Trainable (LoRA): {lora.n_trainable():,}  ({lora.n_trainable()/lora.n_total():.1%} of total)')
print(f'  Initial loss    : {loss_history[0]:.6f}')
print(f'  Final loss      : {loss_history[-1]:.6f}')

In [ ]:
# 3.4  Fine-Tuning Strategy Comparison
D_FT = 64
strats = [
    {'name': 'Full fine-tune',    'rank': D_FT,  'pct': 1.00},
    {'name': 'LoRA rank=16',       'rank': 16,    'pct': None},
    {'name': 'LoRA rank=8',        'rank': 8,     'pct': None},
    {'name': 'LoRA rank=4',        'rank': 4,     'pct': None},
    {'name': 'Prompt tuning (4t)', 'rank': None,  'pct': None},
]

print('Fine-Tuning Strategy Comparison (d_in=d_out=64):')
print(f'{"Strategy":25s} | {"Trainable params":16s} | {"% of Full"}')
print('-'*55)
for s in strats:
    if s['rank'] and s['rank'] <= D_FT:
        n_train = 2 * D_FT * s['rank']  # A + B
        n_full  = D_FT * D_FT
        pct     = n_train / n_full
    elif s['name'].startswith('Prompt'):
        n_train = 4 * D_FT   # 4 soft tokens × d_model
        n_full  = D_FT * D_FT
        pct     = n_train / n_full
    else:
        n_train = D_FT * D_FT
        pct     = 1.0
    print(f'{s["name"]:25s} | {n_train:16,} | {pct:.2%}')

---
## Lunch Break — 1:00–2:00 PM
1. What is the difference between self-attention and cross-attention?
2. Why does Pre-LN (norm before sublayer) converge more stably than Post-LN?
3. How does LoRA reduce trainable parameters without hurting model quality?
4. Why does the decoder need a causal mask during training?
---

## Session 5 — Mini Transformer Capstone: Sequence Classification
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Tokenise Tabular Data as a Sequence of Feature Tokens
# Each of the 30 features in Breast Cancer becomes one token of dimension D.
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)
sc   = StandardScaler().fit(X_tr)
Xtr_sc = sc.transform(X_tr)
Xte_sc = sc.transform(X_te)

D_TF   = 16    # embedding dimension
N_FEAT = Xtr_sc.shape[1]   # 30 features → 30 tokens
N_HEAD_TF = 4
N_LAYER_TF = 2

rng_tf = np.random.default_rng(42)

# Feature embedding: project each scalar feature to D_TF-dimensional vector
# W_emb shape: (n_features, D_TF)  — one embedding vector per feature
W_emb = rng_tf.normal(0, 0.05, (1, D_TF))  # shared scalar → vector projection
# Each sample x is (n_feat,) → tokens are x[:,None]*W_emb → (n_feat, D_TF)

# Add feature ID embeddings (learned position-like per-feature embeddings)
Feat_emb = rng_tf.normal(0, 0.05, (N_FEAT, D_TF))  # (30, D_TF)

def tokenise(x_row):
    """Convert one sample (n_feat,) → token matrix (n_feat, D_TF)."""
    scalar_embed = x_row[:, None] * W_emb   # (30, D_TF) — scale the base vector
    return scalar_embed + Feat_emb           # add feature identity embedding

X_tok = tokenise(Xtr_sc[0])   # one sample for demo
print(f'Input sample shape    : {Xtr_sc[0].shape}  (30 features)')
print(f'Token sequence shape  : {X_tok.shape}  (30 tokens × {D_TF}D)')

In [ ]:
# 5.2  Mini Transformer Classifier
class MiniTransformerClassifier:
    """
    Architecture:
    features → token embeddings → N encoder blocks
    → mean pooling → linear classification head
    """
    def __init__(self, n_feat, d_model, n_heads, n_layers, n_classes, seed=42):
        rng_tc = np.random.default_rng(seed)
        self.n_feat   = n_feat
        self.d_model  = d_model
        self.W_emb    = rng_tc.normal(0, 0.05, (1, d_model))          # scalar projection
        self.Feat_emb = rng_tc.normal(0, 0.05, (n_feat, d_model))     # feature identity
        self.blocks   = [TransformerEncoderBlock(d_model, n_heads, seed=seed+i*10)
                         for i in range(n_layers)]
        self.W_cls    = rng_tc.normal(0, 0.05, (d_model, n_classes))  # classification head
        self.b_cls    = np.zeros(n_classes)

    def embed(self, x_row):
        return x_row[:, None] * self.W_emb + self.Feat_emb

    def forward(self, x_row):
        h = self.embed(x_row)            # (n_feat, d_model)
        attn_maps = []
        for block in self.blocks:
            h, w = block.forward(h)
            attn_maps.append(w)
        pooled = h.mean(axis=0)          # mean pooling over feature tokens
        logits = pooled @ self.W_cls + self.b_cls
        probs  = softmax(logits)
        return probs, pooled, attn_maps

    def predict(self, X):
        preds = []
        for x in X:
            probs, _, _ = self.forward(x)
            preds.append(probs.argmax())
        return np.array(preds)

model_tf = MiniTransformerClassifier(
    n_feat=N_FEAT, d_model=D_TF, n_heads=N_HEAD_TF,
    n_layers=N_LAYER_TF, n_classes=2
)

# Quick forward pass
probs_demo, pooled_demo, attn_demo = model_tf.forward(Xtr_sc[0])
print(f'Mini Transformer Classifier:')
print(f'  Token sequence: ({N_FEAT}, {D_TF})')
print(f'  Pooled rep    : {pooled_demo.shape}')
print(f'  Class probs   : {probs_demo.round(4)}')
print(f'  Attention maps: {len(attn_demo)} blocks × {attn_demo[0].shape}')

In [ ]:
# 5.3  Training Loop (Gradient-Free: Evolutionary Strategy for speed)
# True backprop through numpy Transformer is slow; we use ES as a fast proxy
# that still demonstrates convergence
rng_es = np.random.default_rng(42)

def cross_entropy(probs, label):
    return -np.log(probs[label] + 1e-9)

def evaluate_model(model, X, y):
    """Accuracy on dataset."""
    preds = model.predict(X)
    return accuracy_score(y, preds)

# --- Train with simple gradient approx on the classification head only ---
# This keeps runtime << 60s by only training W_cls and b_cls
LR_TF  = 0.01
N_EPOCHS = 100
N_BATCH  = 32
tf_losses, tf_accs = [], []

for epoch in range(N_EPOCHS):
    # Mini-batch
    idx   = rng_es.integers(0, len(Xtr_sc), N_BATCH)
    batch_loss = 0
    grad_W = np.zeros_like(model_tf.W_cls)
    grad_b = np.zeros_like(model_tf.b_cls)
    for i in idx:
        probs, pooled, _ = model_tf.forward(Xtr_sc[i])
        label  = y_tr[i]
        loss   = cross_entropy(probs, label)
        batch_loss += loss
        # Gradient of cross-entropy + softmax w.r.t. W_cls
        d_logits      = probs.copy()
        d_logits[label] -= 1
        grad_W += np.outer(pooled, d_logits)
        grad_b += d_logits
    # Update only classification head
    model_tf.W_cls -= LR_TF * grad_W / N_BATCH
    model_tf.b_cls -= LR_TF * grad_b / N_BATCH
    tf_losses.append(batch_loss / N_BATCH)
    if epoch % 10 == 0:
        acc = evaluate_model(model_tf, Xte_sc[:50], y_te[:50])
        tf_accs.append((epoch, acc))

final_acc = evaluate_model(model_tf, Xte_sc, y_te)
print(f'Mini Transformer Training ({N_EPOCHS} epochs, batch={N_BATCH}):')
print(f'  Initial loss : {tf_losses[0]:.4f}')
print(f'  Final loss   : {tf_losses[-1]:.4f}')
print(f'  Test accuracy: {final_acc:.4f}')

In [ ]:
# 5.4  Comparison: Mini Transformer vs Logistic Regression
lr_baseline = LogisticRegression(max_iter=500).fit(Xtr_sc, y_tr)
lr_acc      = lr_baseline.score(Xte_sc, y_te)
tf_acc_full = evaluate_model(model_tf, Xte_sc, y_te)

print(f'Model Comparison on Breast Cancer:')
print(f'  Logistic Regression    : {lr_acc:.4f}')
print(f'  Mini Transformer       : {tf_acc_full:.4f}')
print(f'  Gap                    : {tf_acc_full - lr_acc:+.4f}')
print(f'\nKey insight: Transformer treats each feature as a token,')
print(f'allowing any pair of features to attend to each other directly.')

In [ ]:
# 5.5  Attention Pattern Analysis + Final Visualisation
_, _, attn_maps = model_tf.forward(Xte_sc[0])
feature_names  = load_breast_cancer().feature_names

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Attention map from block 1 head 0
attn_block1_h0 = attn_maps[0][0]   # head 0
im1 = axes[0,0].imshow(attn_block1_h0[:10,:10], cmap='Blues')
axes[0,0].set_title('Attention (Block 1, Head 0 — top 10 features)')
axes[0,0].set_xlabel('Key feature'); axes[0,0].set_ylabel('Query feature')
plt.colorbar(im1, ax=axes[0,0])

# Training loss curve
axes[0,1].plot(tf_losses, color='#534AB7', linewidth=1.8)
axes[0,1].set_title('Mini Transformer Training Loss')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Cross-entropy loss')
axes[0,1].grid(alpha=0.3)

# PE visualisation (compact)
pe_sm = sinusoidal_pe(N_FEAT, D_TF)
im3   = axes[1,0].imshow(pe_sm.T, aspect='auto', cmap='RdBu')
axes[1,0].set_title('Positional Encoding (feature positions)')
axes[1,0].set_xlabel('Feature token position'); axes[1,0].set_ylabel('Embedding dim')
plt.colorbar(im3, ax=axes[1,0])

# Accuracy comparison bar
models_c   = ['Logistic\nRegression', 'Mini\nTransformer']
accs_c     = [lr_acc, tf_acc_full]
colors_c   = ['#888780', '#534AB7']
bars = axes[1,1].bar(models_c, accs_c, color=colors_c, alpha=0.85, width=0.5)
axes[1,1].bar_label(bars, fmt='%.4f', padding=4, fontsize=11)
axes[1,1].set_ylim(0.85, 1.02)
axes[1,1].set_title('Model Comparison: Test Accuracy')
axes[1,1].set_ylabel('Accuracy')
axes[1,1].grid(axis='y', alpha=0.3)

plt.suptitle('Day 24 Capstone: Mini Transformer for Sequence Classification', fontsize=12)
plt.tight_layout(); plt.show()

print('\n=== FINAL RESEARCH REPORT ===')
print(f'  Architecture     : {N_LAYER_TF}-layer Transformer encoder ({D_TF}D, {N_HEAD_TF} heads)')
print(f'  Input treatment  : 30 features → 30 tokens each {D_TF}D')
print(f'  Training epochs  : {N_EPOCHS}')
print(f'  Test accuracy    : {tf_acc_full:.4f}')
print(f'  LoRA efficiency  : rank-4 LoRA uses only {lora.n_trainable()/lora.n_total():.1%} of full params')

---
## Day 24 Summary

| Concept | Skill Gained |
|---|---|
| Scaled dot-product attention | QKᵀ/√d_k with optional additive mask |
| Multi-head attention (MHA) | h parallel heads, concat + project through W_o |
| Feed-forward network (FFN) | Expand 4× with GELU, contract back to d_model |
| Layer normalisation | Normalise across feature dim, Pre-LN for stability |
| Residual connections | x + sublayer(norm(x)) — prevents gradient vanishing |
| Sinusoidal PE | sin/cos at 10000^(2i/d) frequencies |
| GELU activation | Smooth ReLU: x·σ(1.702x), used in GPT/BERT |
| Cross-attention | Q from decoder, K/V from encoder memory |
| Causal mask | Upper-triangular -inf prevents future peeking |
| LoRA fine-tuning | W_eff = W + (α/r)·A@B, only 12.5% params |
| Mini transformer | Feature tokenisation → encoder → mean pool → classify |

---
## Day 25 Preview
- Diffusion models: forward noise process and score matching
- DDPM denoising: reverse process with learned noise prediction
- Classifier-free guidance and conditional generation
- Tabular diffusion for synthetic data generation
- Capstone: train a mini diffusion model from scratch

In [ ]:
# Bonus: Day 24 in one cell
rng_b = np.random.default_rng(0)
SEQ_B, D_B, H_B = 6, 16, 4
X_b   = rng_b.normal(0, 1, (SEQ_B, D_B))
mha_b = MultiHeadAttention(D_B, H_B, seed=0)
out_b, w_b = mha_b.forward(X_b, X_b, X_b)
print(f'MHA: {X_b.shape} → {out_b.shape}  head weights: {w_b.shape}')

pe_b = sinusoidal_pe(SEQ_B, D_B)
print(f'Sinusoidal PE: {pe_b.shape}  min={pe_b.min():.3f}  max={pe_b.max():.3f}')

ff_b = FeedForward(D_B, seed=0)
ff_out = ff_b.forward(out_b)
print(f'FFN: {out_b.shape} → {ff_out.shape}  (via {D_B*4}D hidden)')

lr_b = LogisticRegression(max_iter=500).fit(Xtr_sc, y_tr)
print(f'LR baseline acc: {lr_b.score(Xte_sc, y_te):.4f}')
print(f'Mini TF test acc: {tf_acc_full:.4f}')
print(f'LoRA efficiency: {lora.n_trainable()/lora.n_total():.1%} trainable params')
print('\nDay 24 complete — Transformer from scratch, Seq2Seq, LoRA, Mini LM Capstone!')